In [ ]:
#serach by creation date without local filter

import os
import requests
from dotenv import load_dotenv
from datetime import datetime

# === 1️⃣ Load GitHub token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ No GITHUB_TOKEN found in All_Tokens.env")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# === 2️⃣ Base query ===
base_query = "stars:>50 topic:android fork:false archived:false"

# === 3️⃣ Add date range ===
# Example: 2022-01-01 to 2022-12-31
start_date = "2020-04-07"
end_date = "2020-04-10"
date_range = f"created:{start_date}..{end_date}"

# Full query
search_query = f"{base_query} {date_range}"

print(f"🔍 Query: {search_query}")

# === 4️⃣ Call GitHub Search API ===
params = {
    "q": search_query,
    "per_page": 10  # limit for testing
}

response = requests.get(
    "https://api.github.com/search/repositories",
    headers=HEADERS,
    params=params
)

if response.status_code == 200:
    data = response.json()
    print(f"✅ Total matched repos: {data.get('total_count')}")
    for repo in data.get("items", []):
        print(f"- {repo['full_name']} | Stars: {repo['stargazers_count']} | Created: {repo['created_at']}")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)


🔍 Query: stars:>50 topic:android fork:false archived:false created:2020-04-07..2020-04-10
✅ Total matched repos: 18
- florisboard/florisboard | Stars: 7115 | Created: 2020-04-08T21:18:23Z
- DantSu/ESCPOS-ThermalPrinter-Android | Stars: 1355 | Created: 2020-04-07T06:42:00Z
- kongpf8848/Animation | Stars: 562 | Created: 2020-04-08T07:55:57Z
- yuvraj24/react-native-stories-view | Stars: 419 | Created: 2020-04-08T19:34:50Z
- Perfomer/blitz | Stars: 246 | Created: 2020-04-10T17:26:16Z
- realabbas/Github-Actions-React-Native | Stars: 167 | Created: 2020-04-10T14:37:26Z
- Krosxx/yyets_flutter | Stars: 166 | Created: 2020-04-09T09:48:44Z
- thebiltheory/react-native-number-please | Stars: 151 | Created: 2020-04-09T08:02:09Z
- nandan-desai-extras/PrivacyBreacher | Stars: 141 | Created: 2020-04-09T10:56:55Z
- ganeshrvel/flutter_mobx_dio_boilerplate | Stars: 98 | Created: 2020-04-10T09:54:45Z


In [4]:
# search in two layers


import os
import requests
from dotenv import load_dotenv
from datetime import datetime

# === Load GitHub token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ No GITHUB_TOKEN found in All_Tokens.env")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# === Define filter function ===
def filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
    min_stars = 50
    if item.get('stargazers_count', 0) <= min_stars:
        return False
    if item.get('fork', False) != expected_fork:
        return False
    if item.get('archived', False) != expected_archived:
        return False

    language = (item.get('language') or '').lower()
    topics = [t.lower() for t in item.get('topics', [])]
    description = (item.get('description') or '').lower()
    name = (item.get('name') or '').lower()

    if expected_language:
        if language != expected_language:
            return False
    if expected_topic:
        if expected_topic not in topics:
            return False
    if expected_keyword:
        if expected_keyword not in description and expected_keyword not in name:
            return False

    return True

# === Config ===
base_query = "stars:>50 topic:android fork:false archived:false"
start_date = "2020-04-07"
end_date = "2020-04-10"
date_range = f"created:{start_date}..{end_date}"

search_query = f"{base_query} {date_range}"
print(f"🔍 Query: {search_query}")

# === Expected conditions for local filter ===
expected_fork = False
expected_archived = False
expected_language = ''     # because this query has no explicit language
expected_topic = 'android' # we do want topic:android
expected_keyword = ''      # no keyword pattern in this query

# === Call GitHub Search API ===
params = {
    "q": search_query,
    "per_page": 10  # limit for testing
}

response = requests.get(
    "https://api.github.com/search/repositories",
    headers=HEADERS,
    params=params
)

if response.status_code == 200:
    data = response.json()
    print(f"✅ API matched: {data.get('total_count')} repos")
    raw_items = data.get("items", [])
    print(f"🔍 Checking with local filter...")

    passed_items = []
    for item in raw_items:
        if filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
            passed_items.append(item)

    print(f"✅ Passed local filter: {len(passed_items)} of {len(raw_items)}")
    for repo in passed_items:
        print(f"- {repo['full_name']} | Stars: {repo['stargazers_count']} | Topics: {repo.get('topics', [])}")
else:
    print(f"❌ API Error: {response.status_code}")
    print(response.text)


🔍 Query: stars:>50 topic:android fork:false archived:false created:2020-04-07..2020-04-10
✅ API matched: 18 repos
🔍 Checking with local filter...
✅ Passed local filter: 10 of 10
- florisboard/florisboard | Stars: 7115 | Topics: ['android', 'input-method', 'keyboard', 'kotlin', 'kotlin-android']
- DantSu/ESCPOS-ThermalPrinter-Android | Stars: 1355 | Topics: ['android', 'android-library', 'barcode', 'bluetooth-printer', 'escpos', 'escpos-printer', 'qrcode', 'tcp-printer', 'thermal-printer', 'thermal-printing', 'usb-printer']
- kongpf8848/Animation | Stars: 562 | Topics: ['android', 'animation', 'lottie', 'objectanimator', 'sticker', 'svga', 'telegram']
- yuvraj24/react-native-stories-view | Stars: 419 | Topics: ['android', 'component', 'component-library', 'create-react-app', 'cross-platform', 'ios', 'javascript', 'js', 'native', 'props', 'react', 'react-dom', 'react-hooks', 'react-native', 'render', 'types', 'typescript', 'useeffect', 'usereducer', 'usestate']
- Perfomer/blitz | Stars: 